# Arrow Tables vs Datasets

In [ ]:
! pip install pyarrow
! pip install pandas

## Imports

In [ ]:
import pyarrow.csv as csv

## Charger une table en mémoire avec Arrow DataFrame

In [ ]:
csv_file = "data1.csv"
table = csv.read_csv(csv_file)

# Trier les données par une colonne spécifique
sorted_table = table.sort_by("name")
print(sorted_table)

## Travailler sur un dataset qui dépasse la mémoire

In [ ]:
import pyarrow.dataset as ds
import pyarrow.compute as pc
import pandas as pd

# Charger un dataset Parquet
parquet_file = "Path/to/parquet" 
dataset = ds.dataset(parquet_file, format="parquet", partitioning="hive")

# Appliquer des filtres et des agrégations
filtered = dataset.filter(pc.field("BEN_SEX_COD") == "1")

### C'est lazy !

In [ ]:
filtered.count_rows()

In [ ]:
filtered.head(20)

### Maintenant une aggregation avec Acero :

In [1]:
import pyarrow.dataset as ds
import pyarrow.acero as acero
import pyarrow.compute as pc

emplacement_dataset = r"C:\Users\afeldmann\Desktop\openamir"

dataset = ds.dataset(
    emplacement_dataset,
    format="parquet",
    partitioning="hive"
)

In [2]:
plan = acero.Declaration(
    "scan",
    acero.ScanNodeOptions(dataset)
)

plan = acero.Declaration(
    "project",
    acero.ProjectNodeOptions(
        expressions=[
            pc.field("FLT_PAI_MNT"),
            pc.field("annee"),
            pc.field("mois"),
        ],
        names=["FLT_PAI_MNT", "annee", "mois"]
    ),
    inputs=[plan]
)

plan = acero.Declaration(
    "aggregate",
    acero.AggregateNodeOptions(
        aggregates=[
            ("FLT_PAI_MNT", "hash_sum", pc.ScalarAggregateOptions(skip_nulls=False, min_count=0), "FLT_PAI_MNT")
        ],
        keys=["annee", "mois"]
    ),
    inputs=[plan]
)

In [3]:
plan

<pyarrow.acero.Declaration>
ExecPlan with 4 nodes:
:ConsumingSinkNode{}
  :GroupByNode{keys=["annee", "mois"], aggregates=[
  	hash_sum(FLT_PAI_MNT, {skip_nulls=false, min_count=0}),
  ]}
    :ProjectNode{projection=[FLT_PAI_MNT, annee, mois]}
      :SourceNode{}

In [ ]:
table = plan.to_table()
print(table)

## Retourner en pandas pour travailler les résultats simplement :

In [ ]:
# Convertir en DataFrame Pandas
pandas_df = table.to_pandas()
print(pandas_df)